# 🏨 Notebook 1 · Hotel Management — Class Design

**Goal.** Learn *how to pick the right classes* for a small hotel booking system by
watching a design evolve from a messy first attempt into clean, single-purpose classes.

We'll pretend we're interviewing and a friend asks:

> "Design a system that lets guests book rooms in a hotel. It must prevent
> double-booking and compute the bill."

Before writing any code, two skills matter more than syntax:

1. **Find the nouns.** They usually become *classes* (Hotel, Room, Guest, Reservation).
2. **Find the verbs.** They usually become *methods* (search, reserve, cancel, checkout).

### ASCII UML (what we're aiming at)
```
┌───────┐ 1   * ┌──────┐
│ Hotel │───────│ Room │
└───┬───┘       └──────┘
    │ 1
    │ *
    ▼
┌──────────────┐ *   1 ┌───────┐
│ Reservation  │───────│ Guest │
└──────────────┘       └───────┘
```

### Responsibilities (one sentence each — the SRP test)
- `Room` — knows its **number**, **type**, and **nightly rate**.
- `Guest` — knows **who** the customer is (id, name, contact).
- `Reservation` — knows a **room + guest + date range** and can compute its **total**.
- `Hotel` — owns the **collection** of rooms/reservations and the **availability rules**.

If you cannot describe a class in one sentence, it is probably doing too much. 🧹


## 🛠️ Setup

```bash
cd 07-object-oriented-design/hotel-management
uv sync
```

Then pick the `.venv` kernel in the VS Code kernel picker (top-right of the notebook).
If it does not appear: `Cmd+Shift+P` → **Reload Window**.


## 🧪 Attempt 0 — "just use dicts" (the bad version)

Every project starts here. No classes, just dicts and loose functions.
It runs, but notice how much *logic about the hotel* leaks into the caller.


In [1]:
from datetime import date

# Rooms and reservations are just dicts sitting in module-level lists.
rooms_bad = [
    {"number": 101, "type": "SINGLE", "rate": 100},
    {"number": 201, "type": "DOUBLE", "rate": 150},
]
reservations_bad = []  # global mutable state 😬

def book_bad(room_number, guest_name, check_in, check_out):
    # Bug #1: no validation that check_out > check_in.
    # Bug #2: overlap check is wrong — it only looks at the exact same dates.
    for r in reservations_bad:
        if r["room"] == room_number and r["in"] == check_in:
            raise RuntimeError("already booked")
    reservations_bad.append(
        {"room": room_number, "guest": guest_name, "in": check_in, "out": check_out}
    )

book_bad(101, "Ada",   date(2026, 5, 1), date(2026, 5, 4))
# Overlap that should fail but won't, because the naive check only compares exact dates:
book_bad(101, "Grace", date(2026, 5, 2), date(2026, 5, 3))
print("reservations (buggy):", reservations_bad)


reservations (buggy): [{'room': 101, 'guest': 'Ada', 'in': datetime.date(2026, 5, 1), 'out': datetime.date(2026, 5, 4)}, {'room': 101, 'guest': 'Grace', 'in': datetime.date(2026, 5, 2), 'out': datetime.date(2026, 5, 3)}]


### What's wrong?
- 🔴 **No clear owner** of the rules — any file can append to `reservations_bad`.
- 🔴 **Stringly-typed** room types (`"SINGLE"`) — typos crash at runtime.
- 🔴 **Broken overlap check** — `2026-05-02 → 2026-05-03` conflicts with `2026-05-01 → 2026-05-04` but the code happily books both.
- 🔴 **No concept of a `Reservation`** — we can't ask "what's the total?" without recomputing.

These are exactly the pains that classes exist to solve.


## 🧪 Attempt 1 — introduce classes, still coupled (better)

Here we make proper classes, but the `Hotel` *and* the caller both know about
reservation internals. We'll also fix the overlap bug with one small math trick.


In [2]:
class RoomV1:
    def __init__(self, number, type_, rate):
        self.number = number
        self.type = type_
        self.rate = rate

class ReservationV1:
    def __init__(self, room, guest_name, check_in, check_out):
        self.room = room
        self.guest_name = guest_name
        self.check_in = check_in
        self.check_out = check_out

class HotelV1:
    def __init__(self, rooms):
        self.rooms = rooms
        self.reservations = []

    def book(self, room_number, guest_name, check_in, check_out):
        room = next(r for r in self.rooms if r.number == room_number)
        # Correct interval-overlap test: [a_in, a_out) ∩ [b_in, b_out) ≠ ∅
        for r in self.reservations:
            if r.room.number == room_number and (check_in < r.check_out and r.check_in < check_out):
                raise RuntimeError("room busy for those dates")
        res = ReservationV1(room, guest_name, check_in, check_out)
        self.reservations.append(res)
        return res

hotel = HotelV1([RoomV1(101, "SINGLE", 100), RoomV1(201, "DOUBLE", 150)])
hotel.book(101, "Ada", date(2026, 5, 1), date(2026, 5, 4))
try:
    hotel.book(101, "Grace", date(2026, 5, 2), date(2026, 5, 3))  # now correctly rejected
except RuntimeError as e:
    print("blocked ✅:", e)


blocked ✅: room busy for those dates


### Still imperfect
- 🟡 Room *type* is a string, not a type-safe value.
- 🟡 `Reservation` has **no behaviour** (can't compute nights/total).
- 🟡 `Guest` isn't a real concept yet — we pass a loose `guest_name`.
- 🟡 Lots of boilerplate in `__init__`.


## ✅ Attempt 2 — clean OOD (the best version)

Three upgrades make this production-shaped while staying tiny:

1. **`Enum`** for `RoomType` — typos are impossible and each type carries its rate.
2. **`@dataclass`** — Python auto-generates `__init__` / `__repr__` / `__eq__`.
3. **Behaviour lives with data** — `Reservation.nights` and `Reservation.total`
   are *computed properties*, not fields the caller must keep in sync.


In [3]:
from dataclasses import dataclass
from enum import Enum

class RoomType(Enum):
    SINGLE = 100   # value = nightly rate in $
    DOUBLE = 150
    SUITE  = 300

@dataclass(frozen=True)  # frozen => immutable, hashable → safe to put in sets/dicts
class Room:
    number: int
    type: RoomType

@dataclass(frozen=True)
class Guest:
    id: str
    name: str
    email: str = ""

@dataclass
class Reservation:
    id: int
    room: Room
    guest: Guest
    check_in: date
    check_out: date
    cancelled: bool = False

    @property
    def nights(self) -> int:
        return (self.check_out - self.check_in).days

    @property
    def total(self) -> int:
        return self.nights * self.room.type.value

# Quick demo — notice how readable the domain becomes.
ada = Guest("G1", "Ada", "ada@ex.com")
r = Reservation(1, Room(101, RoomType.SINGLE), ada, date(2026, 5, 1), date(2026, 5, 4))
print(r)
print(f"{r.nights} nights  ·  total ${r.total}")


Reservation(id=1, room=Room(number=101, type=<RoomType.SINGLE: 100>), guest=Guest(id='G1', name='Ada', email='ada@ex.com'), check_in=datetime.date(2026, 5, 1), check_out=datetime.date(2026, 5, 4), cancelled=False)
3 nights  ·  total $300


### Why this is the "best" shape at this stage
- 🟢 **Self-describing types**: `RoomType.SUITE` is typo-proof and rate-aware.
- 🟢 **Data + behaviour together**: asking a `Reservation` for its `total` is trivial.
- 🟢 **Immutable value objects** (`Room`, `Guest`): safe to share, compare, hash.
- 🟢 **Small surface area**: each class fits in your head.

The `Hotel` class — which *orchestrates* rooms and reservations — gets its own
notebook next, because orchestration deserves a bad→best walkthrough of its own.


## 🧠 Try it yourself
1. Add a `DELUXE` variant to `RoomType` with rate 220. Create a `Room(401, RoomType.DELUXE)`.
2. Add a `weekend_multiplier` so weekend nights cost 1.25× — where should that logic live?
   *(Hint: start by adding a method on `Reservation`; only promote it to `Hotel` if more than one reservation needs it.)*
3. Which class should own a `cancel()` method — `Reservation` or `Hotel`? Argue both sides.

### Key takeaways
- Find nouns → classes, verbs → methods.
- Keep each class describable in one sentence (Single-Responsibility Principle).
- Prefer `Enum` over strings and `@dataclass` over hand-rolled `__init__`.
- Put behaviour next to the data it uses (`nights`, `total`).
